# 04.决策树初筛_不含评分

使用 LightGBM 做产品/变量初筛，不引入 score 变量。

In [ ]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
SRC_DIR = PROJECT_ROOT / "src"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))

from 函数代码包 import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: "%.6f" % x)

customer_type = "demo_existing_customer"
target_type = "y2"   # 可切换为 "y1"

join_keys = ["id", "cell", "name", "apply_date"]
keep_cols = ["month", "flag"]
target = "y"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("customer_type:", customer_type)
print("target_type:", target_type)

import lightgbm as lgb
from sklearn.model_selection import train_test_split

In [ ]:
fea_info = pd.read_csv(OUTPUT_DIR / f"3A.单变量初筛结果_score全保留_{customer_type}_{target_type}.csv")
data_df = pd.read_csv(OUTPUT_DIR / f"3A.单变量初筛数据_score全保留_{customer_type}_{target_type}.csv").fillna(-999)

train_valid_data = data_df[data_df["flag"].isin(["train", "valid"])].copy()
oot_data = data_df[data_df["flag"] == "oot"].copy()

fea_info["prod"] = fea_info["var"].apply(get_product_prefix)

# 不含评分：剔除 score
fea_cols = fea_info[fea_info["dtype"].isin(["数值型", "类别型"])]["var"].tolist()
fea_cols = [c for c in fea_cols if c in train_valid_data.columns and not c.startswith("score")]

col_vars = fea_cols
cat_lst = [c for c in col_vars if str(train_valid_data[c].dtype) in ["object", "category"]]

for col in cat_lst:
    all_values = pd.concat([train_valid_data[col], oot_data[col]], axis=0).astype(str)
    categories = all_values.astype("category").cat.categories
    train_valid_data[col] = pd.Categorical(train_valid_data[col].astype(str), categories=categories).codes
    oot_data[col] = pd.Categorical(oot_data[col].astype(str), categories=categories).codes

print(len(col_vars), len(cat_lst))

In [ ]:
params = {
    "feature_fraction": 0.6,
    "bagging_fraction": 0.7,
    "bagging_freq": 1,
    "n_estimators": 200,
    "learning_rate": 0.003,
    "num_leaves": 4,
    "lambda_l1": 120,
    "lambda_l2": 150,
    "nthread": 7,
    "boosting_type": "gbdt",
    "is_unbalance": True,
    "metric": "auc",
    "objective": "binary",
    "min_data_in_leaf": 500,
    "verbosity": -1
}

good_rands = []
var_info_df = pd.DataFrame()
max_rand = 20
target_good_num = 1
decay_limit = 0.06
baseline_ks = {"y1": 0.23, "y2": 0.20}
target_ks = baseline_ks.get(target_type, 0.20)

for rand in range(1, max_rand + 1):
    X_train, X_test, y_train, y_test = train_test_split(
        train_valid_data[col_vars],
        train_valid_data[target],
        test_size=0.3,
        random_state=rand,
        stratify=train_valid_data[["month", target]]
    )

    train_set = lgb.Dataset(X_train, y_train, categorical_feature=cat_lst)
    gbm = lgb.train(params, train_set)

    imp = pd.DataFrame({
        "feature_name": gbm.feature_name(),
        "feature_importance": gbm.feature_importance(importance_type="gain"),
        "rand": rand
    })
    var_info_df = pd.concat([var_info_df, imp], axis=0)

    result_train = auc_gini_ks(gbm.predict(X_train), y_train, "train")
    result_test = auc_gini_ks(gbm.predict(X_test), y_test, "test")
    result_oot = auc_gini_ks(gbm.predict(oot_data[col_vars]), oot_data[target], "oot")

    train_ks = result_train["ks"].iloc[0]
    test_ks = result_test["ks"].iloc[0]
    oot_ks = result_oot["ks"].iloc[0]

    print(rand, train_ks, test_ks, oot_ks)

    if abs(train_ks - test_ks) <= decay_limit and abs(train_ks - oot_ks) <= decay_limit and oot_ks >= target_ks:
        good_rands.append(rand)
        if len(good_rands) >= target_good_num:
            break

if not good_rands:
    good_rands = [1]

print("good_rands:", good_rands)

In [ ]:
selected_var_info = var_info_df[var_info_df["rand"].isin(good_rands)].copy()
selected_var_info["prod"] = selected_var_info["feature_name"].apply(get_product_prefix)

selected_stat_info = selected_var_info.groupby(["prod", "feature_name"])[["feature_importance"]].mean().reset_index()
selected_stat_info.to_csv(OUTPUT_DIR / f"4.决策树初筛_产品阶段_不含评分_{customer_type}_{target_type}.csv", index=False)

prod_limit = 15
selected_prod_lst = (
    selected_stat_info[selected_stat_info["feature_importance"] > 0]
    .groupby("prod")["feature_importance"].sum()
    .sort_values(ascending=False)
    .head(prod_limit)
    .index.tolist()
)

rest_col_vars = selected_stat_info[selected_stat_info["prod"].isin(selected_prod_lst)]["feature_name"].tolist()
print(selected_prod_lst)
print(len(rest_col_vars))

In [ ]:
rest_var_info_df = pd.DataFrame()

for rand in good_rands:
    X_train, X_test, y_train, y_test = train_test_split(
        train_valid_data[rest_col_vars],
        train_valid_data[target],
        test_size=0.3,
        random_state=rand,
        stratify=train_valid_data[["month", target]]
    )

    train_set = lgb.Dataset(X_train, y_train)
    gbm = lgb.train(params, train_set)

    imp = pd.DataFrame({
        "feature_name": gbm.feature_name(),
        "feature_importance": gbm.feature_importance(importance_type="gain"),
        "rand": rand
    })
    rest_var_info_df = pd.concat([rest_var_info_df, imp], axis=0)

rest_var_info_df["prod"] = rest_var_info_df["feature_name"].apply(get_product_prefix)
rest_stat_info = rest_var_info_df.groupby(["prod", "feature_name"])[["feature_importance"]].mean().reset_index()
rest_stat_info.to_csv(OUTPUT_DIR / f"4.决策树初筛_初筛结果_不含评分_{customer_type}_{target_type}.csv", index=False)

final_vars = rest_stat_info[rest_stat_info["feature_importance"] > 0]["feature_name"].tolist()

final_train_valid_data = train_valid_data[join_keys + ["flag", target] + final_vars].copy()
final_oot_data = oot_data[join_keys + ["flag", target] + final_vars].copy()

final_train_valid_data.to_csv(OUTPUT_DIR / f"4.train_valid_data_不含评分_{customer_type}_{target_type}.csv", index=False)
final_oot_data.to_csv(OUTPUT_DIR / f"4.oot_data_不含评分_{customer_type}_{target_type}.csv", index=False)

print(final_train_valid_data.shape, final_oot_data.shape)